# A4. Units audit and dimensional-consistency check of the lead equations (review item R1, closes C7)

Question: what units do the variables really have, and are the two lead equations dimensionally consistent?

Findings (all computed below):
1. **Bogotá O3 (MDA8), NO and CO come from the hourly exports (`USA_Ozone/DATA_ORIGINAL`) and are mixing ratios
   (ppb, ppb, ppm), not mass concentrations.** Evidence: for stations whose SISAIRE (µg m⁻³) and hourly series are the
   same measurement, SISAIRE / hourly = 1.96 (O3) and 1.88 (NO2), i.e. exactly M/24.45 g mol⁻¹ (25 °C, 1 atm) with ppb.
   Bogotá NO2 in the master is the SISAIRE network mean (µg m⁻³; master / SISAIRE = 1.01, r = 0.99).
   The published monthly OSMDA8 means of Bustos et al. (2026) (15–45 µg m⁻³) equal ours × ~1.5 (µg m⁻³ per ppb at Bogotá's
   751 hPa, 15 °C), not ours.
2. **Medellín O3 is in µg m⁻³** (Hourmean network mean / SISAIRE Antioquia = 1.03, r = 0.92); Medellín NO and NO2 are assumed µg m⁻³ (not verified).
   So Bogotá and Medellín O3 are **not in the same unit** (ppb vs µg m⁻³).
3. The dimensionally constrained PySR run assumed gases and PM2.5 are all mass concentrations (kg m⁻³). That was
   wrong for CO, NO and O3 in Bogotá. Below, both lead equations are re-checked symbolically under the TRUE units, with each
   fitted constant free to carry whatever unit makes the expression consistent (constants are unit-flexible in PySR).

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import re, unicodedata
import numpy as np, pandas as pd, sympy as sp
from style import ROOT, DATA_OUT
DATA = ROOT / "Data/processed/colombia"
HOURLY = Path("/path/to/home/Documents/Postdoc_MSCA-Bridge/USA_Ozone/DATA_ORIGINAL")   # Bogotá hourly exports (ppb / ppm)
SAVE = True

In [2]:
# 1. Bogotá: SISAIRE (µg m-3) against the hourly exports, same stations, same days
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper(); return re.sub(r"[^A-Z]", "", s)
def hourly_daily(f):
    o = pd.read_csv(HOURLY / f, encoding="utf-8-sig"); o["t"] = pd.to_datetime(o.FECHA, format="%d/%m/%Y %H:%M", errors="coerce")
    d = o.dropna(subset=["t"]).set_index("t").drop(columns="FECHA").apply(pd.to_numeric, errors="coerce").resample("D").mean()
    d.columns = [norm(c) for c in d.columns]; return d
def sisaire(f):
    s = pd.read_csv(ROOT / "Data/COLOMBIA" / f); s.columns = ["est", "d0", "d1", "v"]; s["est"] = s.est.map(norm); s["d0"] = pd.to_datetime(s.d0)
    return s.pivot_table(index="d0", columns="est", values="v")
rows = []
for pol, hf, sf, mw in (("O3", "O3.csv", "BG_O3_reporte_sisaire.csv", 48.00), ("NO2", "NO2.csv", "BG_NO2_reporte_sisaire.csv", 46.01)):
    h, s = hourly_daily(hf), sisaire(sf); cols = sorted(set(h.columns) & set(s.columns)); idx = h.index.intersection(s.index)
    a, b = h.loc[idx, cols], s.loc[idx, cols]; ok = a.notna() & b.notna()
    ratio = (b.where(ok).mean() / a.where(ok).mean())
    for st, r in ratio.items():
        if np.isfinite(r): rows.append(dict(pollutant=pol, station=st, sisaire_over_hourly=round(float(r), 3), expected_ppb_to_ugm3_at_25C_1atm=round(mw / 24.45, 3)))
ratio_tab = pd.DataFrame(rows)
print(ratio_tab.groupby("pollutant").sisaire_over_hourly.describe().round(2))
ratio_tab.pivot(index="station", columns="pollutant", values="sisaire_over_hourly")

           count  mean   std   min   25%   50%   75%   max
pollutant                                                 
NO2         17.0  1.67  0.27  1.29  1.37  1.88  1.88  1.88
O3          15.0  1.59  0.28  1.33  1.39  1.44  1.96  1.97


pollutant,NO2,O3
station,,
BOLIVIA,1.880,1.962
BOSA,1.884,NaN
CENTRODEALTORENDIMIENTO,1.304,1.387
CIUDADBOLIVAR,1.880,1.961
COLINA,1.881,1.963
GUAYMARAL,1.293,1.336
KENNEDY,1.427,1.560
LASFERIAS,1.369,1.391
MINAMBIENTE,1.881,1.410


In [3]:
# 2. Which source feeds the Bogotá master? (NO2 -> SISAIRE µg m-3;  O3, NO, CO -> hourly exports)
b = pd.read_parquet(DATA / "bogota/bogota_o3_master_dataset.parquet"); b["date"] = pd.to_datetime(b.date); b = b.set_index("date")
sd = sisaire("BG_NO2_reporte_sisaire.csv").mean(axis=1); hd = hourly_daily("NO2.csv").mean(axis=1)
idx = b.index.intersection(sd.index).intersection(hd.index); x = b.loc[idx, "no2"]
src = []
for nm, ser in (("SISAIRE µg m-3, network mean", sd), ("hourly export (ppb), network mean", hd)):
    y = ser.loc[idx]; ok = x.notna() & y.notna()
    src.append(dict(master_variable="no2", candidate_source=nm, ratio_master_over_source=round(float(x[ok].mean() / y[ok].mean()), 3), corr=round(float(np.corrcoef(x[ok], y[ok])[0, 1]), 3), n=int(ok.sum())))
pd.DataFrame(src)

,master_variable,candidate_source,ratio_master_over_source,corr,n
0,no2,"SISAIRE µg m-3, network mean",1.011,0.989,2922
1,no2,"hourly export (ppb), network mean",1.318,0.622,2920


In [4]:
# 3. Bustos et al. (2026): monthly OSMDA8 means, our O3 against theirs
o = pd.read_csv(HOURLY / "O3.csv", encoding="utf-8-sig"); o["t"] = pd.to_datetime(o.FECHA, format="%d/%m/%Y %H:%M", errors="coerce")
o = o.dropna(subset=["t"]).set_index("t").drop(columns="FECHA").apply(pd.to_numeric, errors="coerce")
mda8 = o.rolling(8, min_periods=6).mean().resample("D").max()
sel = ["FONTIBON", "KENNEDY", "LAS FERIAS", "USAQUEN", "SAN CRISTOBAL", "PUENTE ARANDA", "MIN AMBIENTE"]     # stations named by Bustos et al. for the seasonal cycle
mon = mda8[sel].groupby(mda8.index.month).mean().mean(axis=1)
P_HPA, T_C = 751.0, 14.9                                            # Bogotá mean surface pressure and temperature
F_LOCAL = 48.00 * P_HPA * 100 / (8.314 * (T_C + 273.15)) * 1e-3    # µg m-3 per ppb of O3 at local conditions
cmp_ = pd.DataFrame({"our_MDA8_ppb (mean of 7 stations)": mon.round(1), "x local factor = µg m-3": (mon * F_LOCAL).round(1)})
paper = {"Jan-Mar": "35-45", "Apr-Jul": "15-25 (lows)", "Sep-Oct": "30-40"}
print(f"1 ppb O3 = {F_LOCAL:.2f} µg m-3 at {P_HPA:.0f} hPa, {T_C} °C.  Bustos et al. (2026): {paper}")
print("ours in µg m-3 (x local factor): Jan-Mar %.0f, Apr-Jul min %.0f, Sep-Oct %.0f" % ((mon[[1, 2, 3]] * F_LOCAL).mean(), (mon[[4, 5, 6, 7]] * F_LOCAL).min(), (mon[[9, 10]] * F_LOCAL).mean()))
cmp_

1 ppb O3 = 1.51 µg m-3 at 751 hPa, 14.9 °C.  Bustos et al. (2026): {'Jan-Mar': '35-45', 'Apr-Jul': '15-25 (lows)', 'Sep-Oct': '30-40'}
ours in µg m-3 (x local factor): Jan-Mar 39, Apr-Jul min 22, Sep-Oct 35


,our_MDA8_ppb (mean of 7 stations),x local factor = µg m-3
t,,
1,24.5,36.9
2,26.1,39.3
3,27.0,40.7
4,21.6,32.5
5,15.1,22.8
6,14.4,21.7
7,16.0,24.0
8,19.8,29.7
9,22.7,34.2


In [5]:
# 4. Medellín: Hourmean network mean against SISAIRE Antioquia (same unit?)
rows = []
for pol, hf, sf in (("O3", "O3_hourmean.xlsx", "ANTIOQUIA_O3_reporte_sisaire.csv"), ("NO2", "NO2_hourmean.xlsx", "ANTIOQUIA_NO2_reporte_sisaire.csv")):
    h = pd.read_excel(ROOT / "Data/COLOMBIA/Hourmean" / hf, usecols=["Dates", "Medellin"]); h = h.set_index(pd.to_datetime(h.Dates)).Medellin.resample("D").mean()
    s = pd.read_csv(ROOT / "Data/COLOMBIA" / sf); s.columns = ["est", "d0", "d1", "v"]; s["d0"] = pd.to_datetime(s.d0); sdd = s.groupby("d0").v.mean()
    i = h.index.intersection(sdd.index); ok = h.loc[i].notna() & sdd.loc[i].notna()
    rows.append(dict(pollutant=pol, hourmean_mean=round(float(h.loc[i][ok].mean()), 2), sisaire_antioquia_mean=round(float(sdd.loc[i][ok].mean()), 2),
                     ratio=round(float(sdd.loc[i][ok].mean() / h.loc[i][ok].mean()), 3), corr=round(float(np.corrcoef(h.loc[i][ok], sdd.loc[i][ok])[0, 1]), 3), n=int(ok.sum())))
pd.DataFrame(rows)

,pollutant,hourmean_mean,sisaire_antioquia_mean,ratio,corr,n
0,O3,26.95,27.77,1.03,0.918,1430
1,NO2,24.90,32.61,1.31,0.278,2171


In [6]:
# 5. The unit table (verified units, and what the PySR units run assumed)
UNITS = pd.DataFrame([
    ("pm25", "both", "SISAIRE network mean", "µg m-3", "µg m-3", "verified (regulatory unit)"),
    ("o3 (MDA8)", "Bogotá", "hourly export -> MDA8", "ppb", "µg m-3", "verified by SISAIRE/hourly ratio = 1.96 = 48/24.45; consistent with Bustos et al. after x1.5"),
    ("o3 (MDA8)", "Medellín", "Hourmean network mean (= SISAIRE Antioquia)", "µg m-3", "µg m-3", "verified: ratio to SISAIRE Antioquia 1.03"),
    ("no", "Bogotá", "hourly export", "ppb", "µg m-3", "same export family as NO2.csv (ppb by ratio); not independently verified for NO"),
    ("no2", "Bogotá", "SISAIRE network mean", "µg m-3", "µg m-3", "verified: master / SISAIRE = 1.01"),
    ("no / no2", "Medellín", "Hourmean / SISAIRE Antioquia", "µg m-3 (assumed)", "µg m-3", "not verified: Hourmean NO2 / SISAIRE Antioquia = 1.31 with r = 0.28 (the two networks average different stations); SISAIRE is a µg m-3 product"),
    ("co", "Bogotá", "hourly export", "ppm (probable)", "µg m-3", "NOT verified: no SISAIRE CO series to compare; mg m-3 would be a 14 % scale difference (0.88 mg m-3 per ppm at Bogotá)"),
    ("w700", "both", "ERA5", "Pa s-1", "Pa s-1", "ERA5 pressure-coordinate vertical velocity"),
    ("wind_speed_era5", "both", "ERA5", "m s-1", "m s-1", ""),
    ("tcwv", "both", "ERA5", "kg m-2", "kg m-2", ""),
    ("radiation_mean", "both", "ERA5 ssrd/3600", "W m-2", "W m-2", ""),
    ("temp_2m", "both", "station / ERA5", "°C (treated as a temperature difference, dimension Θ)", "Θ", ""),
    ("rh_2m, hotspots (count)", "both", "", "dimensionless", "dimensionless", ""),
    ("photochemical_potential", "both", "no2 x radiation_mean", "µg m-3 W m-2", "kg m-3 W m-2", "compound variable; O3 lead equation uses it"),
], columns=["variable", "city", "source", "verified_unit", "unit_assumed_in_PySR_units_run", "evidence"])
if SAVE: UNITS.to_csv(DATA_OUT / "units_audit.csv", index=False)
UNITS

,variable,city,source,verified_unit,unit_assumed_in_PySR_units_run,evidence
0,pm25,both,SISAIRE network mean,µg m-3,µg m-3,verified (regulatory unit)
1,o3 (MDA8),Bogotá,hourly export -> MDA8,ppb,µg m-3,verified by SISAIRE/hourly ratio = 1.96 = 48/2...
2,o3 (MDA8),Medellín,Hourmean network mean (= SISAIRE Antioquia),µg m-3,µg m-3,verified: ratio to SISAIRE Antioquia 1.03
3,no,Bogotá,hourly export,ppb,µg m-3,same export family as NO2.csv (ppb by ratio); ...
4,no2,Bogotá,SISAIRE network mean,µg m-3,µg m-3,verified: master / SISAIRE = 1.01
5,no / no2,Medellín,Hourmean / SISAIRE Antioquia,µg m-3 (assumed),µg m-3,not verified: Hourmean NO2 / SISAIRE Antioquia...
6,co,Bogotá,hourly export,ppm (probable),µg m-3,NOT verified: no SISAIRE CO series to compare;...
7,w700,both,ERA5,Pa s-1,Pa s-1,ERA5 pressure-coordinate vertical velocity
8,wind_speed_era5,both,ERA5,m s-1,m s-1,
9,tcwv,both,ERA5,kg m-2,kg m-2,


In [7]:
# 6. Symbolic dimensional-consistency check (constants are free: each numeric constant gets an unknown unit vector)
DIMS = ("M", "L", "T", "Th")
def vec(**kw): return sp.Matrix([sp.Rational(kw.get(d, 0)) for d in DIMS])
MASS_CONC = vec(M=1, L=-3); W700 = vec(M=1, L=-1, T=-3); SPEED = vec(L=1, T=-1); TCWV = vec(M=1, L=-2); RAD = vec(M=1, T=-3); ZERO = vec(); TEMP = vec(Th=1)
def unit_sets(city="Bogota"):
    assumed = dict(pm25=MASS_CONC, no=MASS_CONC, no2=MASS_CONC, co=MASS_CONC, o3=MASS_CONC, w700=W700, wind_speed_era5=SPEED, tcwv_lag1=TCWV,
                   hotspots_sector_E_lag3=ZERO, hotspots_sector_N=ZERO, hotspots_sector_E_lag2=ZERO, hotspots_sector_NE_lag2=ZERO, no_ratio=ZERO, rh_2m=ZERO, temp_2m=TEMP, photochemical_potential=MASS_CONC + RAD, radiation_mean=RAD)
    true = dict(assumed); true.update(no=ZERO, co=ZERO, o3=ZERO)                     # Bogotá: NO ppb, CO ppm, O3 ppb (dimensionless mixing ratios)
    true["photochemical_potential"] = MASS_CONC + RAD                                  # no2 (µg m-3) x radiation
    return {"assumed in the PySR units run": assumed, "true units (this audit)": true}

def check(eq, units, target):
    """Returns (consistent: bool, constants: {value: unit-vector or symbolic}, message)."""
    syms = {v: sp.Symbol(v) for v in units}
    expr = sp.parse_expr(eq, local_dict={**syms, "sqrt": sp.sqrt}, evaluate=False)
    eqs, consts, counter = [], {}, [0]
    def U(n):
        if n.is_Symbol: return units[n.name]
        if n.is_Float:
            k = f"c{counter[0]}"; counter[0] += 1; s = sp.Matrix(sp.symbols(f"{k}_M {k}_L {k}_T {k}_Th")); consts[k] = (float(n), s); return s
        if n.is_Number: return ZERO
        if n.is_Add:
            us = [U(a) for a in n.args]
            for u in us[1:]: eqs.extend(list(us[0] - u))
            return us[0]
        if n.is_Mul: return sum((U(a) for a in n.args), ZERO)
        if n.is_Pow:
            b, e = n.args; return U(b) * e if e.is_Number else ZERO
        raise ValueError(f"unhandled node {n}")
    top = U(expr); eqs.extend(list(top - target))
    unknowns = sorted({s for _, v in consts.values() for s in v}, key=str)
    sol = sp.solve(eqs, unknowns, dict=True) if eqs else [{}]
    if not sol: return False, {}, "inconsistent: no unit assignment to the constants makes every + / - / equality valid"
    sol = sol[0]
    out = {}
    for k, (val, s) in consts.items():
        u = s.subs(sol); out[f"{val:g}"] = "".join(f"{d}^{sp.nsimplify(u[i])} " for i, d in enumerate(DIMS) if u[i] != 0).strip() or "dimensionless"
    return True, out, "consistent when the constants carry the units listed"

fr = pd.read_csv(DATA_OUT / "front_r2_recomputed.csv")
# 2026-10-06: the O3 lead equation is now the dimensionally constrained one (units front with T in the pool, c15);
# the former free-form lead (falsification front, c16) is kept as a second O3 row for comparison.
LEADS = [("PM2.5 (units front, c13; lead)", fr[(fr.front == "bogota_pm25_units") & (fr.complexity == 13)].equation.iloc[0], "pm25"),
         ("PM2.5 (free-form front, c13; Table S2)", fr[(fr.front == "bogota_pm25_freeform") & (fr.complexity == 13)].equation.iloc[0], "pm25"),
         ("O3 (units front with PM2.5 and T offered, c15; lead)", fr[(fr.front == "bogota_o3_units_temp") & (fr.complexity == 15)].equation.iloc[0], "o3"),
         ("O3 (free-form front with PM2.5 and T offered, c13; Table S2)", fr[(fr.front == "bogota_o3_falsification_pm") & (fr.complexity == 13)].equation.iloc[0], "o3")]
res = []
for name, eq, tgt in LEADS:
    for label, u in unit_sets().items():
        ok, consts, msg = check(eq, u, u[tgt])
        res.append(dict(equation=name, units=label, dimensionally_consistent=ok, constants_must_carry="; ".join(f"{k}: {v}" for k, v in consts.items()), note=msg))
dim = pd.DataFrame(res)
if SAVE: dim.to_csv(DATA_OUT / "dimensional_check_lead_equations.csv", index=False)
pd.set_option("display.max_colwidth", 200); dim

,equation,units,dimensionally_consistent,constants_must_carry,note
0,"PM2.5 (units front, c13; lead)",assumed in the PySR units run,True,0.104284: M^2 L^-6; 15.6795: L^1/2 T^-1/2,consistent when the constants carry the units listed
1,"PM2.5 (units front, c13; lead)",true units (this audit),True,0.104284: M^2 L^-6; 15.6795: M^1 L^-5/2 T^-1/2,consistent when the constants carry the units listed
2,"PM2.5 (free-form front, c13; Table S2)",assumed in the PySR units run,False,,inconsistent: no unit assignment to the constants makes every + / - / equality valid
3,"PM2.5 (free-form front, c13; Table S2)",true units (this audit),False,,inconsistent: no unit assignment to the constants makes every + / - / equality valid
4,"O3 (units front with PM2.5 and T offered, c15; lead)",assumed in the PySR units run,True,0.10787: M^2 L^-6; 2.03434: M^-1/2 L^9/2 T^-3,consistent when the constants carry the units listed
5,"O3 (units front with PM2.5 and T offered, c15; lead)",true units (this audit),True,0.10787: dimensionless; 2.03434: M^2 L^-3 T^-3,consistent when the constants carry the units listed
6,"O3 (free-form front with PM2.5 and T offered, c13; Table S2)",assumed in the PySR units run,False,,inconsistent: no unit assignment to the constants makes every + / - / equality valid
7,"O3 (free-form front with PM2.5 and T offered, c13; Table S2)",true units (this audit),False,,inconsistent: no unit assignment to the constants makes every + / - / equality valid


In [8]:
# 7. Reading the result
# PM2.5: consistent under either set of units, because CO enters only as  CO + (ventilation term) and every constant can carry
#        the unit that balances it. Under the true units CO is a ppm mixing ratio, so the second factor
#        (0.0617 sqrt(fire) + 15.79) carries the unit of PM2.5: 15.79 µg m-3 per ppm of CO, and b (16.2) carries s^2.
# O3 lead (since 2026-10-06): sqrt(no2/no) * radiation * 0.0702 * (pm25 + 0.756 temp_2m), under a square root -- expected
#        consistent under either set of units, because both additive terms inside carry a constant (0.756 takes the unit
#        that makes T comparable with PM2.5, 0.0702 the one that makes the whole equal O3^2). Confirm in the table above.
# O3 free form (Table S2): NOT consistent under either set of units. The denominator adds  no/pm25 + w700  (a ratio and a
#        vertical velocity) with no constant multiplying either, so no unit assignment to the constants can make it valid.
# 15.79 µg m-3 per ppm of CO = 15.79 / 878 (µg m-3 per (µg m-3 of CO)) with 1 ppm CO = 0.878 mg m-3 at 751 hPa, 15 °C:
co_ug = 28.01 * P_HPA * 100 / (8.314 * (T_C + 273.15))                                # µg m-3 per ppm of CO
print(f"1 ppm CO = {co_ug:.0f} µg m-3 here; the factor 15.79 µg m-3 per ppm is a PM2.5/CO mass ratio of {15.79 / co_ug:.3f}")

1 ppm CO = 878 µg m-3 here; the factor 15.79 µg m-3 per ppm is a PM2.5/CO mass ratio of 0.018


In [9]:
# 8. N1 (review round 2, 2026-09-22): the dimensionally completed O3 form
# The published O3 lead equation implicitly fixes two scale constants to 1 (see the dimensional check above):
#   published:  O3 = temp_2m / (no/pm25 + e + w700)              + sqrt((pp/f) / rh)
#   completed:  O3 = a*temp_2m / (no/pm25 + b + c*w700)          + sqrt(pp / (d*rh))
# Fitted the same way as the lead equations (notebook A2): nonlinear least squares on n-1 years, scored
# leave-one-year-out on the held-out year, constants refitted per fold. Reproduces the round-2 check
# (a=0.97, b=0.58, c=0.87, d=0.46; completed pooled R2 0.603 vs published 0.610) so it is auditable.
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

def o3_published(X, e, f):
    temp_2m, no, pm25, w700, pp, rh = X
    return temp_2m / ((no / pm25 + e) + w700) + np.sqrt(np.clip((pp / f) / rh, 0, None))

def o3_completed(X, a, b, c, d):
    temp_2m, no, pm25, w700, pp, rh = X
    return a * temp_2m / ((no / pm25 + b) + c * w700) + np.sqrt(np.clip(pp / (d * rh), 0, None))

O3_FEATS = ["temp_2m", "no", "pm25", "w700", "photochemical_potential", "rh_2m"]
d_o3 = pd.read_parquet(DATA / "bogota/bogota_o3_master_dataset.parquet")
d_o3["date"] = pd.to_datetime(d_o3["date"]); d_o3["year"] = d_o3["date"].dt.year
d_o3 = d_o3.dropna(subset=O3_FEATS + ["o3"]).reset_index(drop=True)
X_o3 = tuple(d_o3[c].to_numpy(float) for c in O3_FEATS); y_o3 = d_o3["o3"].to_numpy(float)

rows = []
for name, model, p0 in (("published (2 constants, a=c=1 fixed)", o3_published, [0.6341454, 0.48264104]),
                        ("completed (4 free constants)", o3_completed, [1.0, 0.6341454, 1.0, 0.48264104])):
    popt_full, _ = curve_fit(model, X_o3, y_o3, p0=p0, maxfev=50000)
    pred = np.full(len(d_o3), np.nan)
    for yr in sorted(d_o3.year.unique()):
        te = (d_o3.year == yr).to_numpy()
        try: popt, _ = curve_fit(model, tuple(a[~te] for a in X_o3), y_o3[~te], p0=popt_full, maxfev=50000)
        except RuntimeError: popt = popt_full
        pred[te] = model(tuple(a[te] for a in X_o3), *popt)
    rows.append(dict(form=name, constants=", ".join(f"{v:.3f}" for v in popt_full), pooled_r2=r2_score(y_o3, pred), n=len(d_o3)))
o3_forms = pd.DataFrame(rows)
if SAVE: o3_forms.to_csv(DATA_OUT / "o3_completed_form.csv", index=False)
print("O3 = a*T/((NO/PM2.5) + b + c*w700) + sqrt(pp/(d*RH)), O3 in ppb, pp = photochemical_potential = no2*radiation_mean")
o3_forms

O3 = a*T/((NO/PM2.5) + b + c*w700) + sqrt(pp/(d*RH)), O3 in ppb, pp = photochemical_potential = no2*radiation_mean


,form,constants,pooled_r2,n
0,"published (2 constants, a=c=1 fixed)","0.625, 0.466",0.610298,2919
1,completed (4 free constants),"0.972, 0.583, 0.871, 0.462",0.603242,2919
